<a href="https://colab.research.google.com/github/kameda-yoshinari/IMISToolExeA/blob/main/800/802_miniGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8-2. From your backpropagation to a mini-GPT


---
This course material was made with the help / advise of Claude (opus).

---


### What you will learn
In 8-1, you wrote a neural network's **backward pass by hand** and verified it with a gradient check.
In this 8-2, three things happen:

1. **The bridge.** We rebuild 8-1's exact MLP in PyTorch, call `loss.backward()`, and confirm
   PyTorch's `autograd` produces the *same gradients you wrote by hand*. The framework is not magic —
   it is your chain rule, automated.
2. **A new task and a new layer.** We move from classifying digits to **predicting the next character**
   in text, and we meet the one idea behind modern language models: **self-attention**.
3. **A mini-GPT.** We assemble a small Transformer, train it on Shakespeare (and then your favorite novel like Wagahai-wa-Nekodearu 吾輩は猫である / Natsume-Souseki 夏目漱石), generate text, and measure
   **how much faster a GPU is than a CPU** for the same model.

### How this week is graded
- **Task cells** marked 🔧 are your work.
- Your `causal_self_attention` must match a reference **fingerprint**.
- Your mini-GPT must reach **validation loss < 2.0**.
- You submit a **CPU vs GPU timing table** and short **written answers**.

### Slots
- **Slot 1:** the bridge + character language modeling + a bigram baseline.
- **Slot 2:** self-attention (Task 2.2) + assemble & train the mini-GPT + CPU/GPU comparison.
- Watch the recommended "Let's build GPT" segment on attention *before* Slot 2.

> The full training run finishes in **~3 min on CPU**, or **~1 min on a free Colab T4 GPU** — you should run in both ways. First run on CPU, then challenge to T4-GPU for the fast path.

## Preparation at Google Colab

All the files will be placed on your Google Drive.

In [ ]:
!echo "Start mounting your Google Drive."
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!echo "Make a working folder and move to there."
%cd /content/drive/My\ Drive/
%mkdir -p IMIS_Tool-A/Work800
%cd       IMIS_Tool-A/Work800
!ls

## 0. Setup

We use pytorch here.

In [ ]:
import time, math
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

# Colab with a T4 runtime -> "cuda". A local Apple-Silicon Mac -> "mps" (its own GPU).
# Otherwise -> "cpu". The rest of the notebook is unchanged by this choice.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("PyTorch", torch.__version__, "| device =", device)
torch.manual_seed(1337)

## 1. The bridge — your hand-written gradients == PyTorch autograd

Below is a compact version of **8-1's** MLP forward+backward (same math you implemented).
We then build the **identical** network in PyTorch, run `autograd`, and compare. If the two agree
to ~`1e-16` (machine precision), you have proof that `autograd` is exactly the backprop you wrote.

In [ ]:
rng = np.random.default_rng(0)
D, Hd, Cc = 8, 16, 4                     # tiny sizes just to demonstrate the match
x_np = rng.normal(size=(32, D)); y_np = rng.integers(0, Cc, size=32)
W1 = rng.normal(0, np.sqrt(2/D),  (D, Hd)); b1 = np.zeros(Hd)
W2 = rng.normal(0, np.sqrt(2/Hd), (Hd, Cc)); b2 = np.zeros(Cc)

# --- hand-written forward + backward (last week) ---
z1 = x_np @ W1 + b1; a1 = np.maximum(0, z1); z2 = a1 @ W2 + b2
ex = np.exp(z2 - z2.max(1, keepdims=True)); probs = ex / ex.sum(1, keepdims=True)
n = 32
dz2 = probs.copy(); dz2[np.arange(n), y_np] -= 1; dz2 /= n
dW2_hand = a1.T @ dz2
dz1 = (dz2 @ W2.T) * (z1 > 0)
dW1_hand = x_np.T @ dz1

# --- same network in PyTorch, gradients via autograd ---
tW1 = torch.tensor(W1, requires_grad=True); tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True); tb2 = torch.tensor(b2, requires_grad=True)
tx = torch.tensor(x_np); ty = torch.tensor(y_np)
logits = torch.relu(tx @ tW1 + tb1) @ tW2 + tb2
loss = F.cross_entropy(logits, ty)       # softmax + cross-entropy + everything, in one call
loss.backward()                          # <-- this replaces ALL of last week's backward code

print("max |autograd - hand-written|:")
print(f"  dW1: {np.abs(tW1.grad.numpy() - dW1_hand).max():.2e}")
print(f"  dW2: {np.abs(tW2.grad.numpy() - dW2_hand).max():.2e}")
print("=> identical to machine precision. autograd IS your chain rule, automated.")

## 2. A new task: predicting the next character

We now model **text**. The idea is the same cross-entropy classification as last week, but the
"classes" are the next character, and we make one prediction at every position in a sequence.

### English source

tiny Shapespare  
https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

### Japanese source

Aozora bunko (青空文庫)  
https://www.aozora.gr.jp  
吾輩は猫である  
https://www.aozora.gr.jp/cards/000148/files/789_14547.html

By default we start with a **English** corpus: tiny Shakespeare.  

Then we can later on try a **Japanese** corpus: Natsume Sōseki's *I Am a Cat* (吾輩は猫である, 1905), which is in the public domain (via Aozora Bunko). Character-level modeling works **identically** for Japanese — the only real difference is that the "alphabet" is now ~3,000 characters (kanji + kana) instead of ~65,
which we will see changes the *scale* of the loss. Set `CORPUS = "en"` to use English Shakespeare instead; nothing else in the notebook changes.

In [ ]:
import os, urllib.request

CORPUS = "en"    # "en" = English (tiny Shakespeare) | "ja" = Japanese (Sōseki, public domain)
                 # The main run below is English, as the text above and the written answers
                 # A/B/C of Task 2.3 ask for. The Japanese corpus is used in Section 6.

SOURCES = {
    "ja": ("wagahai_soseki.txt",
           "https://raw.githubusercontent.com/kameda-yoshinari/IMISToolExeA/main/800/wagahai_soseki.txt"),
    "en": ("shakespeare.txt",
           "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"),
}
fname, url = SOURCES[CORPUS]
for path in (fname, f"./data/{fname}"):   # prefer a local copy (after git clone)
    if os.path.exists(path):
        fname = path; break
else:
    urllib.request.urlretrieve(url, fname)
text = open(fname, encoding="utf-8").read()
print(f"corpus = {CORPUS!r}  |  {len(text):,} characters")
print(text[:180])

**Tokenization (character-level).** We map each unique character to an integer. This is the simplest
possible "tokenizer".  

For English, this yields ~65 tokens. Then we split the data into a training and a validation stream.  
For Japanese, this yields ~3,000 tokens (every distinct kanji and kana is its own class);



In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join(itos[i] for i in l)
print("vocab size:", vocab_size)
print("encode('Hi') =", encode("Hi"), "-> decode ->", decode(encode("Hi")))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]

block_size = 256     # context length: how many characters the model sees to predict the next one
batch_size = 64      # (enlarged from 64/32 - see the note under 'Assemble the mini-GPT')

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("input batch:", xb.shape, "target batch:", yb.shape)

### A baseline with no attention (bigram)

Before attention, here is the crudest model: predict the next character using **only the current one**
(a lookup table). It learns letter frequencies but cannot use context — its samples are gibberish.
This is the bar that attention needs to beat.

In [ ]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.table(idx)
        if targets is None:
            return logits, None
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

bg = Bigram().to(device)
opt = torch.optim.AdamW(bg.parameters(), lr=1e-2)
for _ in range(1000):
    xb, yb = get_batch("train"); _, loss = bg(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
print(f"bigram val-ish loss ~ {loss.item():.3f}  (random guessing would be ln(vocab)={math.log(vocab_size):.3f})")

@torch.no_grad()
def sample(model, n_tokens=200):
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    for _ in range(n_tokens):
        logits, _ = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())
print("--- bigram sample (expect gibberish) ---")
print(sample(bg))

The line above prints the loss of the **last training batch**. The grading criterion compares against `bigram_val_loss`, so we measure the baseline properly: the average loss of the trained bigram model over 50 batches drawn from the **validation** split. This is the number the mini-GPT has to beat by at least 1.0.

In [ ]:
@torch.no_grad()
def eval_loss(m, split="val", iters=50):
    """Average cross-entropy of a model over `iters` random batches of one split."""
    was_training = m.training
    m.eval()
    losses = torch.zeros(iters)
    for i in range(iters):
        xb, yb = get_batch(split)
        _, l = m(xb, yb)
        losses[i] = l.item()
    m.train(was_training)
    return losses.mean().item()

bigram_val_loss = eval_loss(bg, "val")
print(f"bigram_val_loss = {bigram_val_loss:.3f}   "
      f"(random guessing = ln({vocab_size}) = {math.log(vocab_size):.3f})")
print(f"-> the mini-GPT must reach a validation loss <= {bigram_val_loss - 1.0:.3f}")


## 3. 🔧 Task 2.1 — Self-attention (the one new idea)

Attention lets each position **look back** at earlier positions and pull in information. For a single
head with queries `q`, keys `k`, values `v` (each shape `(B, T, head_size)`):

1. **Scores:** $\text{scores} = \dfrac{q\,k^\top}{\sqrt{d_k}}$   → shape `(B, T, T)`
2. **Causal mask:** set the upper triangle to $-\infty$ so a position cannot see the **future**.
3. **Weights:** softmax over the last dimension.
4. **Output:** weighted sum of values, `weights @ v`.

Implement these four steps. `tril` is a lower-triangular matrix of ones provided for the mask.

In [ ]:
def causal_self_attention(q, k, v, tril):
    # q, k, v: (B, T, head_size);  tril: (T, T) lower-triangular ones
    B, T, hs = q.shape
    # ┏━━ Task 2.1: scaled dot-product attention with a causal mask ━━
    # >>> SOLUTION
    # 1. scores: how much each query matches each key, scaled by 1/sqrt(d_k) so that the
    #    dot products do not grow with head_size and push the softmax into saturation.
    scores = (q @ k.transpose(-2, -1)) * hs ** -0.5           # (B, T, T)
    # 2. causal mask: position i may only attend to positions j <= i, so everything above
    #    the diagonal becomes -inf (tril is sliced in case T < block_size).
    scores = scores.masked_fill(tril[:T, :T] == 0, float("-inf"))
    # 3. softmax over the keys -> each row is a probability distribution over the past.
    weights = F.softmax(scores, dim=-1)                       # (B, T, T)
    # 4. weighted sum of the values.
    out = weights @ v                                         # (B, T, head_size)
    # <<< SOLUTION
    return out


**Self-check (provided).** With a fixed seed, the sum of your attention output must equal the
reference **fingerprint**. If it matches, your Task 2.1 is correct.

In [ ]:
torch.manual_seed(0)
B, T, Cc2, head = 2, 5, 8, 4
xx = torch.randn(B, T, Cc2)
Wk = torch.randn(Cc2, head); Wq = torch.randn(Cc2, head); Wv = torch.randn(Cc2, head)
q_, k_, v_ = xx @ Wq, xx @ Wk, xx @ Wv
tril_check = torch.tril(torch.ones(T, T))
out = causal_self_attention(q_, k_, v_, tril_check)
fp = round(out.sum().item(), 4)
print(f"your fingerprint = {fp}   (reference = 3.3232)  ->  {'PASS ✅' if abs(fp-3.3232)<1e-3 else 'FAIL ❌'}")

## 4. Assemble the mini-GPT

We stack your attention into a small Transformer: multi-head attention + a feed-forward network,
with residual connections and layer norm, repeated a few times. This is a real (tiny) GPT — the same
shape as the models behind modern chatbots, just small. It uses **your** `causal_self_attention`.

In [ ]:
# Sizing note: with the original 128/4/4 setting and 2000 iterations the model reaches a
# validation loss of about 1.68 on tiny Shakespeare. That satisfies 'val < 2.0', but the
# bigram baseline is ~2.50, so the margin is only ~0.83 and the second grading criterion
# ('at least 1.0 lower than bigram_val_loss') is not met no matter how long it trains.
# We therefore scale the model up to the size that can actually clear the bar, and switch
# dropout on - it was declared here but never used in the layers below.
n_embd, n_head, n_layer, dropout = 384, 6, 6, 0.2
head_size = n_embd // n_head

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        return causal_self_attention(self.query(x), self.key(x), self.value(x), self.tril)  # YOUR code

class MultiHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.ReLU(),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHead(); self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd); self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # residual connection
        return x + self.ff(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln(self.blocks(x)); logits = self.head(x)
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

model = MiniGPT().to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")

### Train

We track train/val loss. `max_iters=2000` gives a validation loss around **1.8** and readable
(if quirky) Shakespeare. Increase it for better text if you have time/GPU.  
It will take more than 10 minutes for Shapespeare / 2000 (probably ten times faster on T4).

In [ ]:
@torch.no_grad()
def estimate_loss(iters=50):
    model.eval(); out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = get_batch(split); _, l = model(xb, yb); losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train(); return out

# A bigger model needs a smaller learning rate and more steps. We also drop the learning
# rate once near the end, which buys a few extra hundredths of validation loss for free.
max_iters, eval_interval, lr = 6000, 1000, 3e-4
opt = torch.optim.AdamW(model.parameters(), lr=lr)
hist = []
t0 = time.time()
for it in range(max_iters + 1):
    if it % eval_interval == 0:
        e = estimate_loss(); hist.append((it, e["train"], e["val"]))
        print(f"iter {it:4d} | train {e['train']:.3f} | val {e['val']:.3f} | {time.time()-t0:5.1f}s")
    if it == int(0.75 * max_iters):                 # learning-rate step-down
        for g in opt.param_groups: g["lr"] = lr * 0.3
    xb, yb = get_batch("train"); _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
print(f"\ntrained on {device} in {time.time()-t0:.1f}s | final val loss {hist[-1][2]:.3f}")

its, tr, va = zip(*hist)
plt.figure(figsize=(5,3)); plt.plot(its, tr, label="train"); plt.plot(its, va, label="val")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.title("mini-GPT training"); plt.show()

In [ ]:
print("--- mini-GPT sample ---")
print(sample(model, 400))

### Do we pass the two automatic checks?

In [ ]:
gpt_val_loss = eval_loss(model, "val")
margin = bigram_val_loss - gpt_val_loss
print(f"bigram baseline validation loss : {bigram_val_loss:.3f}")
print(f"mini-GPT validation loss        : {gpt_val_loss:.3f}")
print(f"margin over the baseline        : {margin:.3f}")
print()
print(f"check 1 - validation loss < 2.0        : "
      f"{'PASS' if gpt_val_loss < 2.0 else 'FAIL'}")
print(f"check 2 - at least 1.0 below the bigram: "
      f"{'PASS' if margin >= 1.0 else 'FAIL'}")


## 5. 🔧 Task 2.2 — CPU vs GPU

The cell below times one training step on **whatever device you are on now**. Run this notebook
**twice**: once with *Runtime → Change runtime type → CPU*, once with *T4 GPU*, and fill in the table.

Because this model is small, expect the GPU to win by roughly one order of magnitude — but not more.
Bigger models widen the gap; that is the whole reason GPUs matter for deep learning.

In [ ]:
def time_steps(n_steps=50):
    xb, yb = get_batch("train")
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if device == "cuda": torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / n_steps

print(f"device = {device}:  {time_steps():.1f} ms / training step")

The cell above only measures the device the notebook is currently running on. To get the whole table from a single run, the cell below builds a fresh copy of the **same** model on each device that is actually available here and times it. Devices that do not exist on this runtime are reported as such — no number is guessed.

In [ ]:
def time_on(dev, n_steps=30, warmup=5):
    """Time one full training step (forward + backward + optimizer) of the same model on `dev`."""
    m = MiniGPT().to(dev)
    o = torch.optim.AdamW(m.parameters(), lr=lr)
    xb, yb = get_batch("train")
    xb, yb = xb.to(dev), yb.to(dev)
    def step():
        _, l = m(xb, yb)
        o.zero_grad(set_to_none=True); l.backward(); o.step()
    def sync():
        if dev == "cuda": torch.cuda.synchronize()
        if dev == "mps":  torch.mps.synchronize()
    for _ in range(warmup): step()          # warm-up: allocator / kernel compilation
    sync(); t0 = time.time()
    for _ in range(n_steps): step()
    sync()
    return 1000 * (time.time() - t0) / n_steps

available = {
    "cpu":  True,
    "mps":  torch.backends.mps.is_available(),   # Apple-Silicon GPU (local Mac)
    "cuda": torch.cuda.is_available(),           # NVIDIA GPU (Colab T4)
}
timings = {}
for dev, ok in available.items():
    if ok:
        timings[dev] = time_on(dev)
        print(f"{dev:5s}: {timings[dev]:8.1f} ms / training step")
    else:
        print(f"{dev:5s}: not available on this runtime")

if "cpu" in timings:
    print()
    for dev, ms in timings.items():
        print(f"speedup of {dev:4s} vs cpu = {timings['cpu']/ms:5.2f}x")
if torch.cuda.is_available():
    print()
    print(torch.cuda.get_device_name(0))


**Fill in your measurements (🔧 Task 2.2):**

| device | ms / step | speedup vs CPU |
|--------|-----------|----------------|
| CPU    |  *(fill)* | 1.0×           |
| T4 GPU |  *(fill)* | *(fill)*       |


## 6. Corpus challenge

Then do the same with "ja" with/without GPU.    
If you know well about the corpus, you can put more training time to have a better result.  

You started 8-1 by writing a backward pass by hand. You are ending 8-2 having built — and understood, layer by layer — a working (miniature) version of the architecture behind modern LLMs.

### 6.1 Rebuild Section 2 for the Japanese corpus

As the instructions suggest, we re-run the code of Section 2 with a different text. Nothing about the model changes — the same `MiniGPT`, the same `causal_self_attention`, the same training loop. Only the character vocabulary and the data stream are rebuilt.

In [ ]:
CORPUS = "ja"                      # Natsume Soseki, "I Am a Cat" (public domain, Aozora Bunko)
fname, url = SOURCES[CORPUS]
for path in (fname, f"./data/{fname}"):
    if os.path.exists(path):
        fname = path; break
else:
    urllib.request.urlretrieve(url, fname)
text = open(fname, encoding="utf-8").read()
print(f"corpus = {CORPUS!r}  |  {len(text):,} characters")
print(text[:120])

# --- tokenizer rebuilt on the new alphabet ---
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join(itos[i] for i in l)
print("token size (vocabulary) :", vocab_size)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"train tokens {len(train_data):,} | val tokens {len(val_data):,}")
print("random-guessing loss = ln(vocab) =", f"{math.log(vocab_size):.3f}")


### 6.2 The bigram baseline on Japanese

In [ ]:
bg_ja = Bigram().to(device)
opt_bg = torch.optim.AdamW(bg_ja.parameters(), lr=1e-2)
for _ in range(1000):
    xb, yb = get_batch("train"); _, l = bg_ja(xb, yb)
    opt_bg.zero_grad(set_to_none=True); l.backward(); opt_bg.step()

bigram_val_loss_ja = eval_loss(bg_ja, "val")
print(f"[ja] bigram_val_loss = {bigram_val_loss_ja:.3f}")
print(f"[ja] the mini-GPT must reach <= {bigram_val_loss_ja - 1.0:.3f} for the 1.0 margin")


### 6.3 Train the same mini-GPT on Japanese

The Japanese text is about three times shorter than tiny Shakespeare while its alphabet is about 47 times larger, so we use fewer iterations here (the model starts overfitting much earlier). The training time is measured, so it can be compared with the English run.

In [ ]:
model_ja = MiniGPT().to(device)
print(f"parameters: {sum(p.numel() for p in model_ja.parameters())/1e6:.2f} M "
      f"(larger than the English model only because of the {vocab_size}-way output layer)")

max_iters_ja, eval_interval_ja, lr_ja = 3000, 500, 3e-4
opt_ja = torch.optim.AdamW(model_ja.parameters(), lr=lr_ja)
hist_ja = []
t0 = time.time()
for it in range(max_iters_ja + 1):
    if it % eval_interval_ja == 0:
        tr_l = eval_loss(model_ja, "train", iters=20)
        va_l = eval_loss(model_ja, "val",   iters=20)
        hist_ja.append((it, tr_l, va_l))
        print(f"iter {it:4d} | train {tr_l:.3f} | val {va_l:.3f} | {time.time()-t0:5.1f}s")
    if it == int(0.75 * max_iters_ja):
        for g in opt_ja.param_groups: g["lr"] = lr_ja * 0.3
    xb, yb = get_batch("train"); _, l = model_ja(xb, yb)
    opt_ja.zero_grad(set_to_none=True); l.backward(); opt_ja.step()
train_time_ja = time.time() - t0
print(f"\ntrained on {device} in {train_time_ja:.1f}s | final val loss {hist_ja[-1][2]:.3f}")

its, tr, va = zip(*hist_ja)
plt.figure(figsize=(5,3)); plt.plot(its, tr, label="train"); plt.plot(its, va, label="val")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.title("mini-GPT on Japanese"); plt.show()


### 6.4 Generated Japanese text, timing and the margin check

In [ ]:
print("--- mini-GPT (ja) sample ---")
print(sample(model_ja, 400))


In [ ]:
gpt_val_loss_ja = eval_loss(model_ja, "val")
margin_ja = bigram_val_loss_ja - gpt_val_loss_ja
print(f"[ja] bigram baseline : {bigram_val_loss_ja:.3f}")
print(f"[ja] mini-GPT        : {gpt_val_loss_ja:.3f}")
print(f"[ja] margin          : {margin_ja:.3f}  -> "
      f"{'PASS' if margin_ja >= 1.0 else 'FAIL'} (>= 1.0)")
print(f"[ja] training time   : {train_time_ja:.1f}s on {device} for {max_iters_ja} iterations")

# same per-step timing as Task 2.2, but for the Japanese vocabulary
def time_on_corpus(dev, n_steps=30, warmup=5):
    m = MiniGPT().to(dev)
    o = torch.optim.AdamW(m.parameters(), lr=lr_ja)
    xb, yb = get_batch("train"); xb, yb = xb.to(dev), yb.to(dev)
    def step():
        _, l = m(xb, yb); o.zero_grad(set_to_none=True); l.backward(); o.step()
    def sync():
        if dev == "cuda": torch.cuda.synchronize()
        if dev == "mps":  torch.mps.synchronize()
    for _ in range(warmup): step()
    sync(); t0 = time.time()
    for _ in range(n_steps): step()
    sync(); return 1000 * (time.time() - t0) / n_steps

print()
for dev, ok in available.items():
    print(f"[ja] {dev:5s}: {time_on_corpus(dev):8.1f} ms / step" if ok
          else f"[ja] {dev:5s}: not available on this runtime")


---
## 7. 🔧 Task 2.3-D — my own corpus: Arabic song lyrics

For the "your own corpus" part of Task 2.3-D I use **`arabic_songs.txt`**, a collection of Arabic
song lyrics I assembled myself (383,177 characters, 14,259 lines). It is a genuinely different
writing system from both of the corpora provided with the lesson — Arabic script, written
**right-to-left**, with short vowels usually omitted — and it also has a *structure*: every song
starts with `عنوان الأغنية` (song title), then `كلمات الأغنية` (lyrics), then numbered
`مقطع` (section) and `بيت` (line) markers. That structure is a nice test of attention, because
getting it right requires remembering what happened dozens of characters ago, which the bigram
model provably cannot do.

The interesting contrast with Japanese is the **token size**. Japanese needed ~3,000 classes because
every kanji is its own character; Arabic is alphabetic, so this corpus has only **50 distinct
characters** — even fewer than tiny Shakespeare's 65. The loss values are therefore on a comparable
scale to the English run rather than to the Japanese one.

> If you run this notebook on Colab, upload `arabic_songs.txt` first (the file browser on the left,
> or `from google.colab import files; files.upload()`), or place it in the
> `IMIS_Tool-A/Work800` folder on your Drive. It is my own file, so it is not on the course repository.

### 7.1 Load the corpus and rebuild the tokenizer

In [ ]:
CORPUS = "ar"
SOURCES["ar"] = ("arabic_songs.txt", None)   # my own corpus: no download URL

fname, _ = SOURCES[CORPUS]
candidates = (fname, f"./data/{fname}", f"/content/drive/My Drive/IMIS_Tool-A/Work800/{fname}")
for path in candidates:
    if os.path.exists(path):
        fname = path; break
else:
    raise FileNotFoundError(
        "arabic_songs.txt was not found. Upload it next to this notebook "
        f"(looked in: {candidates})")

text = open(fname, encoding="utf-8").read()
print(f"corpus = {CORPUS!r}  |  {len(text):,} characters  |  {text.count(chr(10)):,} lines")
print(text[:220])

# --- tokenizer rebuilt on the Arabic alphabet ---
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: "".join(itos[i] for i in l)

print()
print("token size (vocabulary):", vocab_size)
print("the alphabet:", "".join(chars).replace(chr(10), " "))
print("random-guessing loss = ln(vocab) =", f"{math.log(vocab_size):.3f}")

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"train tokens {len(train_data):,} | val tokens {len(val_data):,}")


### 7.2 The bigram baseline on Arabic

In [ ]:
bg_ar = Bigram().to(device)
opt_bg = torch.optim.AdamW(bg_ar.parameters(), lr=1e-2)
for _ in range(1000):
    xb, yb = get_batch("train"); _, l = bg_ar(xb, yb)
    opt_bg.zero_grad(set_to_none=True); l.backward(); opt_bg.step()

bigram_val_loss_ar = eval_loss(bg_ar, "val")
print(f"[ar] bigram_val_loss = {bigram_val_loss_ar:.3f}")
print(f"[ar] the mini-GPT must reach <= {bigram_val_loss_ar - 1.0:.3f} for the 1.0 margin")
print()
print("--- bigram sample (expect gibberish) ---")
print(sample(bg_ar, 200))


### 7.3 Train the mini-GPT on the Arabic corpus

Exactly the same `MiniGPT` class and the same `causal_self_attention` — only the data changed.

In [ ]:
model_ar = MiniGPT().to(device)
print(f"parameters: {sum(p.numel() for p in model_ar.parameters())/1e6:.2f} M")

max_iters_ar, eval_interval_ar, lr_ar = 3000, 500, 3e-4
opt_ar = torch.optim.AdamW(model_ar.parameters(), lr=lr_ar)
hist_ar = []
t0 = time.time()
for it in range(max_iters_ar + 1):
    if it % eval_interval_ar == 0:
        tr_l = eval_loss(model_ar, "train", iters=20)
        va_l = eval_loss(model_ar, "val",   iters=20)
        hist_ar.append((it, tr_l, va_l))
        print(f"iter {it:4d} | train {tr_l:.3f} | val {va_l:.3f} | {time.time()-t0:5.1f}s")
    if it == int(0.75 * max_iters_ar):
        for g in opt_ar.param_groups: g["lr"] = lr_ar * 0.3
    xb, yb = get_batch("train"); _, l = model_ar(xb, yb)
    opt_ar.zero_grad(set_to_none=True); l.backward(); opt_ar.step()
train_time_ar = time.time() - t0
print(f"\ntrained on {device} in {train_time_ar:.1f}s | final val loss {hist_ar[-1][2]:.3f}")

its, tr, va = zip(*hist_ar)
plt.figure(figsize=(5,3)); plt.plot(its, tr, label="train"); plt.plot(its, va, label="val")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend()
plt.title("mini-GPT on Arabic song lyrics"); plt.show()


### 7.4 Generated Arabic lyrics

In [ ]:
print("--- mini-GPT (ar) sample ---")
print(sample(model_ar, 600))


### 7.5 Margin check and timing for the Arabic run

In [ ]:
gpt_val_loss_ar = eval_loss(model_ar, "val")
margin_ar = bigram_val_loss_ar - gpt_val_loss_ar
print(f"[ar] bigram baseline : {bigram_val_loss_ar:.3f}")
print(f"[ar] mini-GPT        : {gpt_val_loss_ar:.3f}")
print(f"[ar] margin          : {margin_ar:.3f}  -> "
      f"{'PASS' if margin_ar >= 1.0 else 'FAIL'} (>= 1.0)")
print(f"[ar] val loss < 2.0  : {'PASS' if gpt_val_loss_ar < 2.0 else 'FAIL'}")
print(f"[ar] training time   : {train_time_ar:.1f}s on {device} "
      f"for {max_iters_ar} iterations")

print()
for dev, ok in available.items():
    print(f"[ar] {dev:5s}: {time_on_corpus(dev):8.1f} ms / step" if ok
          else f"[ar] {dev:5s}: not available on this runtime")

print()
print("summary of all three corpora")
print(f"{'corpus':8s} {'vocab':>6s} {'bigram':>8s} {'miniGPT':>8s} {'margin':>8s}")
for name, vs, b, g in [("en", 65, bigram_val_loss, gpt_val_loss),
                       ("ja", 3039, bigram_val_loss_ja, gpt_val_loss_ja),
                       ("ar", vocab_size, bigram_val_loss_ar, gpt_val_loss_ar)]:
    print(f"{name:8s} {vs:6d} {b:8.3f} {g:8.3f} {b-g:8.3f}")


## Assignment 8-2 — submit this notebook

**Automatic checks (must pass):**
1. Your `causal_self_attention` matches the reference fingerprint (`3.3232`).
2. Your mini-GPT **clearly beats the bigram baseline** — its final validation loss is at least **1.0 lower
   than `bigram_val_loss`** (printed in Section 2). This criterion is corpus-independent: it works whether
   you train on Japanese or English. *(As a rough guide, the Japanese run reaches the mid-single-digits and
   English reaches ~1.8, but only the margin over the baseline is graded.)*

**Deliverables:** the training loss curve, a generated text sample, and the completed CPU/GPU table.

**Written answers (🔧 Task 2.3).** 2–4 sentences each, below.  
**As for A/B/C, the answers are for "en" experiment environment.**

**A.** What does the **causal mask** do, and why is it essential for a *next-character* model?
What would go wrong at training time without it?

**B.** The bigram baseline and the mini-GPT both minimize the same cross-entropy loss. Why does the
mini-GPT produce so much more coherent text?

**C.** From your CPU/GPU table: the GPU is faster, but the speedup is "only" ~10×, not 1000×. Why is the gap modest for *this* model, and when would it grow?

**D.** Try your own corpus (you can change language as it accepts UTF-8). Describe the token size. Show the time spent on CPU (or GPU). Show the example output. To obtain nice output, did you need to change iteration size?   
As for the execution verification, you'd better rewrite the code cell of the section 2 (and later if needed) of this jupyter notebook.

**A.** *(your answer here)*

---

**B.** *(your answer here)*

---

**C.** *(your answer here)*

---

**D.** *(your answer here)*

---
---

**X.** *(your student ID)*

---

**Y.** *(your name)*



---
Tools and Practices for Intelligent Interaction Systems A  
Master's and Docotal programs in intelligent and mechanical interaction systems, University of Tsukuba, Japan.  
KAMEDA Yoshinari, SHIBUYA Takeshi  

知能システムツール演習a  
知能機能システム学位プログラム (筑波大学大学院)  
担当：亀田能成，澁谷長史  

2026/07/27. ver.B. 説明追加 / Additional Explanation   
